# Business Entity Resolution: Version 7 (exact-key rules + LightGBM)

**How to run:** click **Save Version** (top right), choose **Save & Run All (Commit)**, then **Save**. Settings: CPU (no accelerator), Internet On. It runs on Kaggle's servers even if you close the tab.

What is new compared with v1:
1. **Keys.** Each record gets short "keys" such as *house number + street word + one word of the name*, or *the whole name + house number*. Spelling is made robust first (so "healthcare" and the Hindi "हेल्थकेयर" give the same key).
2. **Extra shortlist.** Pairs that share a key are added even when v1's shortlist missed them.
3. **Look-alike checks.** The data contains planted look-alikes: same name and street but a different house number (712 vs 719) or a different legal form (LLP vs Private Limited). New checks compare house numbers exactly and compare legal forms (v1 threw legal words away).
4. **Rules.** On training data we measure how often each key rule is right. Rules that are right at least 99.5% of the time can accept a pair directly.
5. **LightGBM** (same settings as v1) sees all of the above as extra information.
6. **One business per record:** every Source 2/3 record is given only to its single best business.

The training cell prints `BEST val F0.5` on the same held-out businesses as v1 (v1 = 0.9529; this version scored **0.9647** in our test run). It also tries "model only" vs "rules + model" and keeps whichever scores higher (in our run, "model only" won by a hair: the model already learned what the rules know).

Cells: 1 install, 2 v1 pipeline code, 3 key code, 4 settings, 5 **train**, 6 **predict** test and write `output/matching_results.tsv`, 7 official format check.
Expect about 1 hour for training and 2.5 to 4 hours for prediction. No external data or APIs are used, only the competition files.

In [ ]:
!pip install -q rapidfuzz sparse_dot_topn unidecode polars lightgbm

In [ ]:
# =====================================================================
# Business Entity Resolution: full pipeline (clean -> shortlist -> features -> LightGBM -> decide)
# No external data or services are used anywhere. Only the provided files.
# =====================================================================
import os, re, gc, glob, time, json
import numpy as np
import pandas as pd
import polars as pl
from multiprocessing import Pool
from unidecode import unidecode
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
from rapidfuzz.process import cpdist
import lightgbm as lgb

N_CPU = os.cpu_count() or 4

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

# ---------------------------------------------------------------------
# 1. Loading. Files are tab-separated, all columns read as text.
#    Handles files split into *_part_N.tsv (BOM / CRLF / repeated header).
# ---------------------------------------------------------------------
def find_file(root, name):
    hits = sorted(glob.glob(f"{root}/**/{name}.tsv", recursive=True))
    if hits:
        return [hits[0]]
    parts = glob.glob(f"{root}/**/{name}_part_*.tsv", recursive=True)
    parts.sort(key=lambda p: int(re.findall(r"_part_(\d+)", p)[-1]))
    if not parts:
        raise FileNotFoundError(f"{name}.tsv not found under {root}")
    return parts

def load(root, name):
    frames, header = [], None
    for p in find_file(root, name):
        txt = open(p, encoding="utf-8-sig").read().replace("\r\n", "\n").replace("\r", "")
        if txt.startswith(("entity_id\t", "source1_entity_id\t")):
            header, txt = txt.split("\n", 1)
        frames.append(txt)
    body = "".join(f if f.endswith("\n") else f + "\n" for f in frames)
    df = pl.read_csv((header + "\n" + body).encode(), separator="\t", quote_char=None, infer_schema_length=0)
    return df.fill_null("").to_pandas()

# ---------------------------------------------------------------------
# 2. Cleaning (normalisation). Hand-written rules only.
# ---------------------------------------------------------------------
ABBR = {"rd":"road","st":"street","ave":"avenue","av":"avenue","blvd":"boulevard","bd":"boulevard","hwy":"highway",
        "dr":"drive","ln":"lane","ct":"court","pl":"place","pkwy":"parkway","cir":"circle","ste":"suite","fl":"floor",
        "apt":"apartment","no":"number","nr":"near","opp":"opposite","bldg":"building","ind":"industrial",
        "dist":"district","r":"rue","ch":"chemin","n":"north","s":"south","e":"east","w":"west","mt":"mount",
        "ft":"fort","hts":"heights","sq":"square","imp":"impasse","all":"allee"}
NULLS = {"null", "na", "none", "nan"}
# legal words, written in "skeleton" form (see skel_tok): covers Limited/Ltd/limittedd, Private/Pvt/praaivett, LLC, SARL ...
LEGAL_SK = {"lmtd","ltd","prwt","pwt","llc","llp","inc","ncrprtd","crp","crprtn","c","cmpny","plc","srl","ss","sa",
            "sc","erl","snc","gmbh","t","m","s","d","dba","lp","pc","pllc","pa","ms","and"}

def base(s):
    s = unidecode(str(s)).lower()                                   # also turns Hindi/Telugu/... script into Latin letters
    s = re.sub(r"<?null>?|n/a", " ", s)
    s = re.sub(r"https?://|www\.|\.(com|net|org|in|co|io|biz|fr|us)\b", " ", s)
    s = s.replace("&", " and ").replace("+", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def skel_tok(t):
    """Spelling-robust 'skeleton' of a word: drop vowels, merge ph/f, bh/b ..., collapse double letters.
    'limited' and the transliterated 'limittedd' both become 'lmtd'."""
    if not t or t.isdigit():
        return t
    t = t.replace("0", "o").replace("1", "l") if re.search(r"[a-z]", t) else t   # leetspeak: C1ean -> Clean
    t = re.sub(r"ph", "f", t)
    t = re.sub(r"([bkgtdcs])h", r"\1", t)
    t = t.replace("v", "w").replace("q", "k").replace("z", "s").replace("x", "ks")
    t = t[0] + re.sub(r"[aeiouy]", "", t[1:])
    return re.sub(r"(.)\1+", r"\1", t)

US = {"alabama":"al","alaska":"ak","arizona":"az","arkansas":"ar","california":"ca","colorado":"co","connecticut":"ct",
"delaware":"de","florida":"fl","georgia":"ga","hawaii":"hi","idaho":"id","illinois":"il","indiana":"in","iowa":"ia",
"kansas":"ks","kentucky":"ky","louisiana":"la","maine":"me","maryland":"md","massachusetts":"ma","michigan":"mi",
"minnesota":"mn","mississippi":"ms","missouri":"mo","montana":"mt","nebraska":"ne","nevada":"nv","new hampshire":"nh",
"new jersey":"nj","new mexico":"nm","new york":"ny","north carolina":"nc","north dakota":"nd","ohio":"oh","oklahoma":"ok",
"oregon":"or","pennsylvania":"pa","rhode island":"ri","south carolina":"sc","south dakota":"sd","tennessee":"tn",
"texas":"tx","utah":"ut","vermont":"vt","virginia":"va","washington":"wa","west virginia":"wv","wisconsin":"wi",
"wyoming":"wy","district of columbia":"dc","puerto rico":"pr"}
IN = {"andhra pradesh":"ap","arunachal pradesh":"ar","assam":"as","bihar":"br","chhattisgarh":"cg","goa":"ga",
"gujarat":"gj","haryana":"hr","himachal pradesh":"hp","jharkhand":"jh","karnataka":"ka","kerala":"kl",
"madhya pradesh":"mp","maharashtra":"mh","manipur":"mn","meghalaya":"ml","mizoram":"mz","nagaland":"nl","odisha":"od",
"orissa":"od","punjab":"pb","rajasthan":"rj","sikkim":"sk","tamil nadu":"tn","telangana":"ts","tripura":"tr",
"uttar pradesh":"up","uttarakhand":"uk","uttaranchal":"uk","west bengal":"wb","delhi":"dl","new delhi":"dl",
"jammu and kashmir":"jk","ladakh":"la","puducherry":"py","pondicherry":"py","chandigarh":"ch",
"andaman and nicobar islands":"an","lakshadweep":"ld"}
MERGE_ST = {"ts": "ap"}            # Telangana was split from Andhra Pradesh; the data mixes them
_STATE = {}
for _d in (US, IN):
    for _full, _code in _d.items():
        _STATE[tuple(skel_tok(t) for t in _full.split())] = _code
_CODES = set(US.values()) | set(IN.values())

def states_of(raw_addr):
    """All state/province codes mentioned in an address ('' when none). Country-agnostic: unknown countries get ''."""
    found = set()
    for seg in unidecode(str(raw_addr)).lower().split(","):
        toks = re.sub(r"[^a-z ]", " ", seg).split()
        if not toks:
            continue
        if len(toks) == 1 and toks[0] in _CODES:
            found.add(toks[0]); continue
        sk = [skel_tok(t) for t in toks]
        for n in (4, 3, 2, 1):
            for i in range(len(sk) - n + 1):
                key = tuple(sk[i:i + n])
                if key in _STATE and (n > 1 or len(key[0]) >= 3):
                    found.add(_STATE[key])
    return "|".join(sorted({MERGE_ST.get(x, x) for x in found}))

def _clean_rows(rows):
    out = []
    for name, addr in rows:
        nb = base(name)
        sk = [skel_tok(t) for t in nb.split()]
        core = [t for t in sk if t not in LEGAL_SK] or sk
        an = " ".join(ABBR.get(t, t) for t in base(addr).split() if t not in NULLS)
        out.append((nb, " ".join(core), "".join(core), an,
                    " ".join(skel_tok(t) for t in an.split()), states_of(addr)))
    return out

def prepare(df):
    """Adds the cleaned columns. Runs on all CPU cores."""
    rows = list(zip(df.business_name.tolist(), df.business_address.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_clean_rows, [rows[i:i + step] for i in range(0, len(rows), step)])
    res = [r for chunk in res for r in chunk]
    cols = ["name_b", "name_sk", "name_skc", "addr_n", "addr_sk", "states"]
    for j, c in enumerate(cols):
        df[c] = [r[j] for r in res]
    df.drop(columns=["business_address"], inplace=True)
    df["nonascii"] = df.business_name.str.contains(r"[^\x00-\x7f]", regex=True).astype(np.int8)
    df.drop(columns=["business_name"], inplace=True)
    return df.reset_index(drop=True)

# ---------------------------------------------------------------------
# 3. Shortlist ("blocking"). For each Source 1 record, look only inside the same country and
#    the same state, and keep the top-k most similar names and the top-k most similar addresses.
#    Records whose state could not be read form a separate pool per country.
# ---------------------------------------------------------------------
def _vec_name():
    return TfidfVectorizer(analyzer="char", ngram_range=(3, 3), dtype=np.float32, sublinear_tf=True)
def _vec_addr():
    return TfidfVectorizer(analyzer="word", token_pattern=r"\S+", ngram_range=(1, 2), dtype=np.float32, sublinear_tf=True)

class Index:
    """TF-IDF indexes over one pool of candidate records (one country of Source 2 or Source 3)."""
    def __init__(self, B, k_state=15, k_unk=10):
        self.B, self.k_state, self.k_unk = B, k_state, k_unk
        self.cache = {}
        st = B.states.str.split("|")
        self.members = {}
        for pos, lst in zip(range(len(B)), st):
            for s in lst:
                if s:
                    self.members.setdefault(s, []).append(pos)
        self.unknown = np.flatnonzero(B.states.to_numpy() == "")

    def _build(self, key, pos):
        if key in self.cache:
            return self.cache[key]
        pos = np.asarray(pos, dtype=np.int64)
        entry = {"pos": pos}
        for col, mk in (("name_skc", _vec_name), ("addr_sk", _vec_addr)):
            try:
                v = mk(); M = v.fit_transform(self.B[col].to_numpy()[pos])
                entry[col] = (v, M.T.tocsr())
            except ValueError:            # empty vocabulary, e.g. every address blank
                entry[col] = None
        self.cache[key] = entry
        return entry

    def query(self, A, thr=0.1, chunk=40_000):
        """Returns DataFrame(a, b, sim_name, sim_addr); a = row in A, b = row in self.B."""
        out = []
        ast = A.states.str.split("|").tolist()
        groups = {}
        for i, lst in enumerate(ast):
            for s in lst:
                groups.setdefault(s, []).append(i)
        jobs = []
        for s, rows in groups.items():
            if s == "":
                jobs.append(("__all__", np.arange(len(self.B)), rows, self.k_state))
            elif s in self.members:
                jobs.append((s, self.members[s], rows, self.k_state))
        known = [i for i, lst in enumerate(ast) if lst != [""]]
        if len(self.unknown) and known:
            jobs.append(("__unknown__", self.unknown, known, self.k_unk))
        for key, pos, rows, k in jobs:
            e = self._build(key, pos)
            rows = np.asarray(rows)
            for col, tag in (("name_skc", 0), ("addr_sk", 1)):
                if e[col] is None:
                    continue
                v, BT = e[col]
                texts = A[col].to_numpy()[rows]
                for i in range(0, len(rows), chunk):
                    C = sp_matmul_topn(v.transform(texts[i:i + chunk]), BT, top_n=k, threshold=thr,
                                       n_threads=N_CPU).tocoo()
                    out.append((rows[i + C.row], e["pos"][C.col], C.data, tag))
        if not out:
            return pd.DataFrame({"a": [], "b": [], "sim_name": [], "sim_addr": []})
        a = np.concatenate([o[0] for o in out]).astype(np.int64)
        b = np.concatenate([o[1] for o in out]).astype(np.int64)
        sim = np.concatenate([o[2] for o in out]).astype(np.float32)
        tag = np.concatenate([np.full(len(o[0]), o[3], np.int8) for o in out])
        df = pd.DataFrame({"key": a * len(self.B) + b,
                           "sim_name": np.where(tag == 0, sim, np.nan),
                           "sim_addr": np.where(tag == 1, sim, np.nan)})
        df = df.groupby("key", sort=False).max().reset_index()
        df["a"] = (df.key // len(self.B)).astype(np.int32)
        df["b"] = (df.key % len(self.B)).astype(np.int32)
        return df.drop(columns="key")

# ---------------------------------------------------------------------
# 4. Features: how similar is each (Source 1, candidate) pair?
# ---------------------------------------------------------------------
FULL = [("tset", fuzz.token_set_ratio), ("tsort", fuzz.token_sort_ratio), ("ratio", fuzz.ratio),
        ("partial", fuzz.partial_ratio), ("jw", JaroWinkler.normalized_similarity)]

def _nums(s):
    return set(re.findall(r"\d+", s))

def features(P, A, B, is_s3):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    g = lambda D, col, idx: D[col].to_numpy()[idx]
    f = pd.DataFrame({"sim_name": P.sim_name.to_numpy(), "sim_addr": P.sim_addr.to_numpy()})
    for col, pre, scorers in (("name_b", "nb", FULL), ("name_sk", "nsk", FULL),
                              ("name_skc", "nskc", [("ratio", fuzz.ratio), ("partial", fuzz.partial_ratio)]),
                              ("addr_sk", "ask", [("tset", fuzz.token_set_ratio), ("ratio", fuzz.ratio),
                                                  ("partial", fuzz.partial_ratio)])):
        x, y = g(A, col, a).tolist(), g(B, col, b).tolist()
        for nm, sc in scorers:
            f[f"{pre}_{nm}"] = cpdist(x, y, scorer=sc, workers=N_CPU).astype(np.float32)
    xa, ya = g(A, "addr_n", a), g(B, "addr_n", b)
    nl = [_nums(s) for s in xa]; nr = [_nums(s) for s in ya]
    f["num_jacc"] = np.array([len(p & q) / len(p | q) if (p | q) else -1 for p, q in zip(nl, nr)], np.float32)
    f["num_contain"] = np.array([len(p & q) / len(q) if q else -1 for p, q in zip(nl, nr)], np.float32)
    sa, sb = g(A, "states", a), g(B, "states", b)
    f["state_rel"] = np.array([-1 if (not p or not q) else (1 if set(p.split("|")) & set(q.split("|")) else 0)
                               for p, q in zip(sa, sb)], np.int8)
    f["name_exact"] = (g(A, "name_skc", a) == g(B, "name_skc", b)).astype(np.int8)
    f["cand_addr_len"] = np.fromiter((len(s) for s in ya), np.int32, len(ya))
    f["s1_addr_len"] = np.fromiter((len(s) for s in xa), np.int32, len(xa))
    f["cand_name_len"] = np.fromiter((len(s) for s in g(B, "name_b", b)), np.int32, len(b))
    f["s1_name_len"] = np.fromiter((len(s) for s in g(A, "name_b", a)), np.int32, len(a))
    f["cand_nonascii"] = g(B, "nonascii", b)
    f["is_s3"] = np.int8(is_s3)
    # context: how this candidate compares with the other candidates of the same Source 1 record
    v = f.nsk_tset.to_numpy() + f.ask_tset.to_numpy()
    grp = pd.DataFrame({"a": a, "v": v}).groupby("a")["v"]
    f["rank_s1"] = grp.rank(ascending=False, method="min").to_numpy(np.float32)
    f["gap_s1"] = (grp.transform("max").to_numpy() - v).astype(np.float32)
    f["n_cand_s1"] = grp.transform("size").to_numpy(np.int32)
    return f

# ---------------------------------------------------------------------
# 5. Scoring (same formula as the leaderboard) and the final decision rule
# ---------------------------------------------------------------------
def macro_f05(pred, truth, ids):
    sc = []
    for s in ids:
        p, t = pred.get(s, set()), truth.get(s, set())
        if not t:
            sc.append(1.0 if not p else 0.0); continue
        tp = len(p & t)
        if tp == 0:
            sc.append(0.0); continue
        P, R = tp / len(p), tp / len(t)
        sc.append(1.25 * P * R / (0.25 * P + R))
    return float(np.mean(sc))

def decide(pairs, thr):
    """pairs: DataFrame(s1_id, cand_id, p). Keep p >= thr; each S2/S3 record goes to at most one S1 (its best)."""
    d = pairs[pairs.p >= thr].sort_values("p", ascending=False).drop_duplicates("cand_id")
    return d.groupby("s1_id")["cand_id"].apply(set).to_dict()

def write_tsv(path, s1_ids, mapping, col):
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(f"source1_entity_id\t{col}\n")
        for s in s1_ids:
            fh.write(s + "\t" + ",".join(sorted(mapping.get(s, ()))) + "\n")


In [ ]:
# =====================================================================
# Version 7: exact "keys" (rule layer)
# A key is a short piece of text built from a record, e.g. "house number + street word + one word of the name".
# Two records that share a rare key are very likely the same business.
# We use keys in three ways:
#   1. extra shortlist: pairs sharing a key are added even if the TF-IDF shortlist missed them
#   2. extra features for LightGBM: which keys matched, and how common each key is
#   3. rules: key types that were (almost) always right on training data accept a pair directly
# Needs the functions from the v1 pipeline cell (prepare, skel_tok, log, N_CPU ...).
# =====================================================================
from hashlib import blake2b
import ctypes

def free_mem():
    """Free memory and hand it back to the system (so helper processes don't copy it)."""
    gc.collect()
    try: ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception: pass

KEY_TYPES = ["name", "nameset", "name_num", "addr", "addr_tok", "tok2_num"]
KCAP = 30                    # keys shared by more than this many records on one side are too common to use

def _ktok(t):
    """Extra-strong spelling key for one word: skeleton, and c/k, w/v, j/z sound the same.
    ('healthcare' and the transliterated Hindi 'helthker' both become 'hltkr')."""
    return skel_tok(t).replace("c", "k").replace("j", "z")

def _h(s):
    return int.from_bytes(blake2b(s.encode(), digest_size=8).digest(), "little", signed=True)

def _keys_rows(args):
    rows, start = args
    ri, kt, kh = [], [], []
    for off, (country, name_sk, addr_n) in enumerate(rows):
        r = start + off
        toks = [_ktok(t) for t in name_sk.split() if t]
        toks = [t for t in toks if t]
        ks = []
        cat = "".join(toks)
        if len(cat) >= 4:
            ks.append((0, cat))
            ks.append((1, "".join(sorted(set(toks)))))
        at = addr_n.split()
        nums = []
        for i, t in enumerate(at):                          # every number in the address + the next word with letters
            d = re.sub(r"\D", "", t)
            if not d or len(nums) >= 3:
                continue
            d = d.lstrip("0") or "0"
            nxt = next((_ktok(x) for x in at[i + 1:i + 4] if re.search(r"[a-z]", x)), "")
            nums.append((d, nxt))
        big = [t for t in toks if len(t) >= 3 and not t.isdigit()][:4]
        for d, nxt in dict.fromkeys(nums):
            if len(cat) >= 4:
                ks.append((2, cat + "|" + d))
            if nxt:
                ks.append((3, d + "|" + nxt))
                for t in big:
                    ks.append((4, d + "|" + nxt + "|" + t))
            if len(big) >= 2:
                ks.append((5, big[0] + "|" + big[1] + "|" + d))
        for t, s in dict.fromkeys(ks):
            ri.append(r); kt.append(t); kh.append(_h(f"{country}|{t}|{s}"))
    return np.array(ri, np.int32), np.array(kt, np.int8), np.array(kh, np.int64)

def make_keys(D):
    """All keys of every record in a prepared table. Returns polars DataFrame(row, kt, kh, cnt)
    where cnt = how many records of this table share the key."""
    rows = list(zip(D.country.tolist(), D.name_sk.tolist(), D.addr_n.tolist()))
    step = 50_000
    with Pool(N_CPU) as pool:
        res = pool.map(_keys_rows, [(rows[i:i + step], i) for i in range(0, len(rows), step)])
    K = pl.DataFrame({"row": np.concatenate([r[0] for r in res]), "kt": np.concatenate([r[1] for r in res]),
                      "kh": np.concatenate([r[2] for r in res])})
    K = K.with_columns(pl.len().over("kh").cast(pl.Int32).alias("cnt"))
    return K

def key_pairs(KA, KB, a_rows=None):
    """Pairs (a, b) that share at least one key, with per key type: hit, how many S1 / candidates share it.
    KA: keys of Source 1 (row = row in the S1 table); KB: keys of the candidate table.
    a_rows: optional array of S1 rows to keep (a batch); output 'a' is then the position inside a_rows."""
    ka = KA.filter(pl.col("cnt") <= KCAP)
    if a_rows is not None:
        pos = pl.DataFrame({"row": np.asarray(a_rows, np.int32), "a": np.arange(len(a_rows), dtype=np.int32)})
        ka = ka.join(pos, on="row").drop("row")
    else:
        ka = ka.rename({"row": "a"})
    kb = KB.filter(pl.col("cnt") <= KCAP).rename({"row": "b", "cnt": "cb"})
    J = ka.rename({"cnt": "ca"}).join(kb.drop("kt"), on="kh")
    if J.height == 0:
        return None
    agg = []
    for t, nm in enumerate(KEY_TYPES):
        m = pl.col("kt") == t
        agg += [m.any().cast(pl.Int8).alias(f"k_{nm}"),
                pl.col("ca").filter(m).min().alias(f"k_{nm}_ca"),
                pl.col("cb").filter(m).min().alias(f"k_{nm}_cb")]
    agg += [(pl.col("kt") == 4).sum().cast(pl.Int16).alias("k_addr_tok_n"), pl.len().cast(pl.Int16).alias("k_n")]
    return J.group_by(["a", "b"]).agg(agg).to_pandas()

KEY_FEATS = [f"k_{nm}{s}" for nm in KEY_TYPES for s in ("", "_ca", "_cb")] + ["k_addr_tok_n", "k_n"]

def merge_key_pairs(P, KP, key_only_types=("name", "nameset", "name_num", "addr_tok", "tok2_num")):
    """Add key features to the TF-IDF shortlist P(a, b, sim_name, sim_addr), and add key-only pairs
    (pairs the TF-IDF shortlist missed) when a name-based key matched."""
    if KP is None or len(KP) == 0:
        for c in KEY_FEATS:
            P[c] = np.nan if c.endswith(("_ca", "_cb")) else 0
        P["from_tfidf"] = np.int8(1)
        return P
    KP = KP.copy()
    rare = [(KP[f"k_{t}"].to_numpy() == 1) & (KP[f"k_{t}_ca"].fillna(1e9).to_numpy() <= 1)
            & (KP[f"k_{t}_cb"].fillna(1e9).to_numpy() <= 3) for t in key_only_types]
    extra = KP[np.logical_or.reduce(rare)]                   # only rare keys may add pairs the shortlist missed
    P = P.merge(KP, on=["a", "b"], how="left")
    P["from_tfidf"] = np.int8(1)
    new = extra.merge(P[["a", "b"]], on=["a", "b"], how="left", indicator=True)
    new = new[new._merge == "left_only"].drop(columns="_merge")
    new["sim_name"] = np.nan; new["sim_addr"] = np.nan; new["from_tfidf"] = np.int8(0)
    P = pd.concat([P, new[P.columns]], ignore_index=True)
    for t in KEY_TYPES:
        P[f"k_{t}"] = P[f"k_{t}"].fillna(0).astype(np.int8)
    P["k_addr_tok_n"] = P["k_addr_tok_n"].fillna(0).astype(np.int16)
    P["k_n"] = P["k_n"].fillna(0).astype(np.int16)
    P["a"] = P.a.astype(np.int64); P["b"] = P.b.astype(np.int64)
    return P

# ---------------------------------------------------------------------
# Exactness checks. The data contains planted look-alikes: same name and street, but a different
# house number (712 vs 719) or a different legal form (LLP vs Private Limited). v1 threw legal words away.
# ---------------------------------------------------------------------
LEGAL_MAP = {"prwt": "pvt", "pwt": "pvt", "prvt": "pvt", "lmtd": "ltd", "ltd": "ltd", "lmt": "ltd", "llp": "llp",
             "llc": "llc", "inc": "inc", "ncrprtd": "inc", "crp": "corp", "crprtn": "corp", "cmpny": "co", "c": "co",
             "plc": "plc", "pc": "pc", "pllc": "pllc", "lp": "lp", "srl": "sarl", "eurl": "eurl", "ss": "sas",
             "ssu": "sas", "sa": "sa", "gmbh": "gmbh", "snc": "snc", "sc": "sci", "sk": "sci", "dba": "dba"}

def _extra_rows(rows):
    out = []
    for nb, an in rows:
        lg = set(LEGAL_MAP.get(skel_tok(t)) for t in nb.split()) - {None}
        if "pvt" in lg: lg.discard("ltd")                  # "Private Limited" is one legal form
        nums = [x.lstrip("0") or "0" for x in re.findall(r"\d+", an)]
        out.append(("|".join(sorted(lg)), " ".join(nums)))
    return out

def add_extra(D):
    """Adds per-record columns: legal (legal form, e.g. 'pvt', 'llp|co') and nums (numbers in the address)."""
    rows = list(zip(D.name_b.tolist(), D.addr_n.tolist())); step = 100_000
    with Pool(N_CPU) as pool:
        res = [r for ch in pool.map(_extra_rows, [rows[i:i + step] for i in range(0, len(rows), step)]) for r in ch]
    D["legal"] = [r[0] for r in res]; D["nums"] = [r[1] for r in res]
    return D

def _near(x, y):
    """Two different numbers that look like a planted change: close in value, or same length and one digit apart."""
    if x == y: return False
    if abs(int(x[:9]) - int(y[:9])) <= 20: return True
    return len(x) == len(y) and sum(p != q for p, q in zip(x, y)) == 1

EXACT_FEATS = ["legal_rel", "legal_s1", "legal_cand", "num_eq", "num_cand_extra", "num_s1_extra", "num_first_eq",
               "num_near", "x_noextra", "x_legal_same"]

def _exact_rows(args):
    out = np.zeros((len(args[0]), 8), np.int16)
    for i, (x, y, p, q) in enumerate(zip(*args)):
        lr = -1 if (not x or not y) else int(x == y)
        pl_, ql = p.split(), q.split(); ps, qs = set(pl_), set(ql)
        ea, eb = ps - qs, qs - ps
        first = -1 if (not pl_ or not ql) else int(pl_[0] == ql[0])
        near = int(any(_near(u, v) for u in ea for v in eb)) if ea and eb else 0
        out[i] = (lr, int(bool(x)), int(bool(y)), int(ps == qs), len(eb), len(ea), first, near)
    return out

def exact_features(P, A, B):
    a, b = P.a.to_numpy(), P.b.to_numpy()
    cols = [A.legal.to_numpy()[a], B.legal.to_numpy()[b], A.nums.to_numpy()[a], B.nums.to_numpy()[b]]
    step = 200_000
    with Pool(N_CPU) as pool:
        res = pool.map(_exact_rows, [[c[i:i + step].tolist() for c in cols] for i in range(0, len(P), step)])
    F = pd.DataFrame(np.concatenate(res) if res else np.zeros((0, 8), np.int16), columns=EXACT_FEATS[:8])
    F["x_noextra"] = ((F.num_cand_extra.to_numpy() == 0) & (pd.Series(cols[3]).str.len().to_numpy() > 0)).astype(np.int8)
    F["x_legal_same"] = (F.legal_rel.to_numpy() == 1).astype(np.int8)
    return F


In [ ]:
ROOT = os.environ.get("ROOT", "/kaggle/input")      # the notebook searches here for the .tsv files
WORK = os.environ.get("WORK", "/kaggle/working")    # model + outputs are saved here
N_TRAIN_S1 = int(os.environ.get("N_TRAIN_S1", 150_000))   # training businesses (same 150k as v1, so scores compare)
BATCH = int(os.environ.get("BATCH", 100_000))       # test businesses processed at once (lower it if memory runs out)
RULE_PREC = 0.995            # a key rule is used on its own only if it was right at least 99.5% of the time
RULE_MIN_N = 200             # ... and it fired at least this many times on training pairs
os.makedirs(WORK, exist_ok=True)
log("CPU cores:", N_CPU)


In [ ]:
# =====================================================================
# Version 7 training
# Same 150k training businesses and the same 30k held-out check businesses as v1 (so the scores compare).
# =====================================================================
def rule_hit(D, rules):
    """True where any accepted key rule fires. A rule is a list of conditions that must ALL hold;
    a condition is [key type, max S1 sharing the key, max candidates sharing the key],
    or [check] for a yes/no check: x_noextra = the candidate has no house number that S1 lacks,
    x_legal_same = both names carry the same legal form (Pvt Ltd, LLP, Inc ...)."""
    hit = np.zeros(len(D), bool)
    for rule in rules:
        h = np.ones(len(D), bool)
        for cond in rule:
            if len(cond) == 1:                                 # a yes/no check, e.g. "x_noextra"
                h &= D[cond[0]].to_numpy() == 1; continue
            nm, ca, cb = cond
            h &= (D[f"k_{nm}"].to_numpy() == 1) & (D[f"k_{nm}_ca"].fillna(1e9).to_numpy() <= ca) \
                 & (D[f"k_{nm}_cb"].fillna(1e9).to_numpy() <= cb)
        hit |= h
    return hit

def with_rules(pv, mode):
    """Final score: in mode 'rules', a pair accepted by a rule gets score 2 (always kept, and it wins
    when one Source 2/3 record is claimed by several businesses)."""
    if mode == "rules":
        return pv.assign(p=np.where(pv.rule.to_numpy(), 2.0, pv.p.to_numpy()))
    return pv

log("loading train Source 1 + ground truth")
s1 = load(ROOT, "train_source1"); gt = load(ROOT, "train_ground_truth")
truth = {r.source1_entity_id: set(x for x in r.matched_entity_ids.split(",") if x) for r in gt.itertuples()}
del gt; gc.collect()
samp = s1.sample(min(N_TRAIN_S1, len(s1)), random_state=0).index.to_numpy()   # same sample as v1
S1 = add_extra(prepare(s1)); del s1; gc.collect(); log("prepared all train S1", S1.shape)
KA = make_keys(S1); log("S1 keys", KA.height)            # keys of ALL S1, so 'how many businesses share a key' is right
KA = KA.filter(pl.col("row").is_in(samp))                  # counts are done; keep only the sampled businesses' keys
A = S1.iloc[samp].reset_index(drop=True); del S1; gc.collect()
TRAIN_PARTS = f"{WORK}/train_parts_v7"; os.makedirs(TRAIN_PARTS, exist_ok=True)
for src, is_s3 in (("train_source2", 0), ("train_source3", 1)):
    if os.path.exists(f"{TRAIN_PARTS}/{src}.done"):
        log(src, "already done"); continue
    log("loading", src); B = add_extra(prepare(load(ROOT, src))); log("prepared", src, B.shape)
    KB = make_keys(B); log(src, "keys", KB.height)
    for country in A.country.unique():
        sel = (A.country == country).to_numpy()
        Ac = A[sel].reset_index(drop=True); bpos = np.flatnonzero((B.country == country).to_numpy())
        if len(Ac) == 0 or len(bpos) == 0: continue
        P = Index(B.iloc[bpos].reset_index(drop=True)).query(Ac)
        P["b"] = bpos[P.b.to_numpy()]                     # position in the full B table
        P = merge_key_pairs(P, key_pairs(KA, KB, samp[sel]))
        F = features(P, Ac, B, is_s3)
        for c in KEY_FEATS + ["from_tfidf"]:
            F[c] = P[c].to_numpy()
        E = exact_features(P, Ac, B)
        for c in EXACT_FEATS:
            F[c] = E[c].to_numpy()
        m = pd.DataFrame({"s1_id": Ac.entity_id.to_numpy()[P.a], "cand_id": B.entity_id.to_numpy()[P.b]})
        m["label"] = [c in truth.get(s, ()) for s, c in zip(m.s1_id, m.cand_id)]
        log(src, country, "pairs", len(P), "of which only from keys", int((P.from_tfidf == 0).sum()),
            "true", int(m.label.sum()), "(keys only:", int(m.label[P.from_tfidf.to_numpy() == 0].sum()), ")")
        # each block is saved to disk and freed, so memory stays low while the next source is cleaned
        F.astype(np.float32).to_parquet(f"{TRAIN_PARTS}/X_{src}_{country}.parquet")
        m.to_parquet(f"{TRAIN_PARTS}/M_{src}_{country}.parquet"); del P, E, F, m; gc.collect()
    open(f"{TRAIN_PARTS}/{src}.done", "w").close()
    del B, KB; free_mem()
del KA; gc.collect()
blocks = sorted(glob.glob(f"{TRAIN_PARTS}/X_*.parquet"))
X = pd.concat([pd.read_parquet(f) for f in blocks], ignore_index=True)
M = pd.concat([pd.read_parquet(f.replace("/X_", "/M_")) for f in blocks], ignore_index=True)
y = M.label.to_numpy().astype(int)
all_true = sum(len(truth[s]) for s in A.entity_id)
log("pairs", len(M), "per S1", round(len(M) / len(A), 1), "| shortlist recall", round(y.sum() / all_true, 4),
    "| without key pairs", round(y[X.from_tfidf.to_numpy() == 1].sum() / all_true, 4))

rng = np.random.default_rng(42)
val_ids = set(rng.choice(A.entity_id.to_numpy(), int(0.2 * len(A)), replace=False))   # same as v1
iv = M.s1_id.isin(val_ids).to_numpy()

# ---- 1. rule layer: measure every key rule on the TRAINING part only ----
log("key rules, measured on training pairs (not the held-out check businesses):")
tr = X[~iv]; ytr = y[~iv]; rules = []
EXTRAS = [[], [["x_noextra"]], [["x_legal_same"]], [["x_noextra"], ["x_legal_same"]]]
cands = [[[nm, ca, cb]] + ex for nm in KEY_TYPES for ca in (1, 2) for cb in (1, 3, 10) for ex in EXTRAS]
cands += [[[n1, 1, cb], [n2, 1, cb]] for i, n1 in enumerate(KEY_TYPES) for n2 in KEY_TYPES[i + 1:] for cb in (1, 3, 10)]
for rule in cands:
    h = rule_hit(tr, [rule]); n = int(h.sum())
    if n == 0: continue
    prec = ytr[h].mean()
    ok = prec >= RULE_PREC and n >= RULE_MIN_N
    desc = " AND ".join(f"{c[0]}(S1<={c[1]},cand<={c[2]})" if len(c) == 3 else c[0] for c in rule)
    print(f"   {desc:66s} fires {n:8d}  right {prec:.4f}" + ("  <- KEEP" if ok else ""), flush=True)
    if ok: rules.append(rule)
h = rule_hit(tr, rules)
log("kept", len(rules), "rules; together they fire", int(h.sum()), "right", round(float(ytr[h].mean()), 4) if h.any() else "-",
    "and find", round(ytr[h].sum() / max(ytr.sum(), 1), 4), "of the true pairs in the shortlist")
del tr, ytr, h; free_mem()

# ---- 2. LightGBM for everything, with the key hits as extra features ----
model = lgb.LGBMClassifier(n_estimators=4000, learning_rate=0.05, num_leaves=255, min_child_samples=100,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, n_jobs=N_CPU, verbose=-1)
model.fit(X[~iv], y[~iv], eval_set=[(X[iv], y[iv])], callbacks=[lgb.early_stopping(100, verbose=False)])
log("trees", model.best_iteration_)
pv = M[iv][["s1_id", "cand_id"]].assign(p=model.predict_proba(X[iv])[:, 1], rule=rule_hit(X[iv], rules))
vl = list(val_ids); vt = {s: truth[s] for s in vl}
log("baseline all-empty", round(macro_f05({}, vt, vl), 4))
log("ceiling (perfect model on this shortlist)",
    round(macro_f05(M[iv & (y == 1)].groupby("s1_id").cand_id.apply(set).to_dict(), vt, vl), 4))
log("rules alone (no model)", round(macro_f05(decide(pv[pv.rule].assign(p=1.0), 0.5), vt, vl), 4))

# ---- 3. decision: model only vs rules + model; each S2/S3 record goes to its single best business ----
res = []
for mode in ("model", "rules"):
    pm = with_rules(pv, mode)
    for t in np.arange(0.30, 0.91, 0.05):
        s = macro_f05(decide(pm, t), vt, vl); res.append((mode, round(t, 2), s))
        log(f"{mode:5s} threshold {t:.2f} val F0.5 {s:.4f}")
best_mode, best_t, best_s = max(res, key=lambda r: r[2])
log("BEST val F0.5", round(best_s, 4), "| decision:", best_mode, "threshold", best_t, "| v1 was 0.9529")

# refit on all sampled S1 with the chosen number of trees
final = lgb.LGBMClassifier(**{**model.get_params(), "n_estimators": model.best_iteration_})
final.fit(X, y)
final.booster_.save_model(f"{WORK}/model_v7.txt")
json.dump({"threshold": best_t, "mode": best_mode, "rules": rules, "val_f05": best_s, "features": list(X.columns)},
          open(f"{WORK}/model_v7_meta.json", "w"))
imp = pd.Series(final.feature_importances_, X.columns).sort_values(ascending=False)
log("top features:\n" + imp.head(15).to_string())
del X, M, y, final, model; free_mem()


In [ ]:
# =====================================================================
# Version 7 test prediction. Each batch is saved to disk (parts_v7/), so a crash can resume.
# =====================================================================
PARTS = f"{WORK}/parts_v7"; os.makedirs(PARTS, exist_ok=True)
meta = json.load(open(f"{WORK}/model_v7_meta.json")); booster = lgb.Booster(model_file=f"{WORK}/model_v7.txt")
log("decision:", meta["mode"], "threshold", meta["threshold"], "| key rules used:", len(meta["rules"]))
log("loading test Source 1"); A = add_extra(prepare(load(ROOT, "test_source1"))); log("prepared", A.shape)
print(A.country.value_counts().to_string())
s1_ids = A.entity_id.to_numpy()
KA = None
for src, is_s3 in (("test_source2", 0), ("test_source3", 1)):
    todo = [(c, i) for c in A.country.unique() for i in range(0, int((A.country == c).sum()), BATCH)
            if not os.path.exists(f"{PARTS}/{src}_{c}_{i}.parquet")]
    if not todo:
        log(src, "already done"); continue
    if KA is None:
        KA = make_keys(A); log("S1 keys", KA.height)
    log("loading", src); B = add_extra(prepare(load(ROOT, src))); log("prepared", src, B.shape)
    KB = make_keys(B); log(src, "keys", KB.height)
    bids = B.entity_id.to_numpy()
    for country in A.country.unique():                 # any label works, France included
        a_rows = np.flatnonzero(A.country.to_numpy() == country)
        bpos = np.flatnonzero((B.country == country).to_numpy())
        if len(bpos) == 0 or not any(c == country for c, _ in todo): continue
        idx = Index(B.iloc[bpos].reset_index(drop=True))
        for i in range(0, len(a_rows), BATCH):
            out = f"{PARTS}/{src}_{country}_{i}.parquet"
            if os.path.exists(out): continue
            rows = a_rows[i:i + BATCH]; Ac = A.iloc[rows].reset_index(drop=True)
            P = idx.query(Ac)
            P["b"] = bpos[P.b.to_numpy()]
            P = merge_key_pairs(P, key_pairs(KA, KB, rows))
            if len(P):
                F = features(P, Ac, B, is_s3)
                for c in KEY_FEATS + ["from_tfidf"]:
                    F[c] = P[c].to_numpy()
                E = exact_features(P, Ac, B)
                for c in EXACT_FEATS:
                    F[c] = E[c].to_numpy()
                p = booster.predict(F[meta["features"]].to_numpy(np.float32), num_threads=N_CPU).astype(np.float32)
                rule = rule_hit(F, meta["rules"])
                del F
            else:
                p = np.zeros(0, np.float32); rule = np.zeros(0, bool)
            pd.DataFrame({"s1_row": rows[P.a.to_numpy()].astype(np.int32), "cand_id": bids[P.b.to_numpy()],
                          "p": p, "rule": rule}).to_parquet(out)
            log(src, country, f"batch {i // BATCH + 1}/{-(-len(a_rows) // BATCH)}", "pairs", len(P),
                "rule hits", int(rule.sum()))
            del P, p; gc.collect()
        del idx; gc.collect()
    del B, KB; free_mem()
del KA; gc.collect()

s1_ids = A.entity_id.to_numpy(); del A; gc.collect()
log("combining batches:", len(glob.glob(f"{PARTS}/*.parquet")), "files")
# 1) Final matches: only pairs that can be chosen are loaded (low memory, same as v1.2)
sel = pl.scan_parquet(f"{PARTS}/*.parquet")
if meta["mode"] == "rules":        # a pair accepted by a key rule gets score 2: always kept, and it wins ties
    sel = sel.with_columns(pl.when(pl.col("rule")).then(2.0).otherwise(pl.col("p")).alias("p"))
best = (sel.filter(pl.col("p") >= meta["threshold"]).collect()
        .sort("p", descending=True).unique(subset="cand_id", keep="first", maintain_order=True))  # each S2/S3 record -> its best S1 only
match = best.group_by("s1_row").agg(pl.col("cand_id").sort())
mline = np.full(len(s1_ids), "", dtype=object)
for r, c in zip(match["s1_row"].to_list(), match["cand_id"].to_list()):
    mline[r] = ",".join(c)
n = int((mline != "").sum()); n_links = best.height; del best, match; gc.collect()
os.makedirs(f"{WORK}/output", exist_ok=True)
with open(f"{WORK}/output/matching_results.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for s_, m in zip(s1_ids, mline):
        f.write(s_ + "\t" + m + "\n")
log("done:", len(s1_ids), "rows,", n, "with matches,", len(s1_ids) - n, "empty; avg matches",
    round(n_links / len(s1_ids), 2))
del mline; gc.collect()

# 2) Candidate list, one batch at a time (the Source 2 and Source 3 files of a batch cover the same S1 rows)
cline = np.full(len(s1_ids), "", dtype=object)
for f2 in sorted(glob.glob(f"{PARTS}/test_source2_*.parquet")):
    files = [f2] + [x for x in [f2.replace("test_source2_", "test_source3_")] if os.path.exists(x)]
    g = pl.concat([pl.read_parquet(x, columns=["s1_row", "cand_id"]) for x in files]) \
          .group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True))
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()):
        cline[r] = ",".join(c)
    del g; gc.collect()
for f3 in sorted(glob.glob(f"{PARTS}/test_source3_*.parquet")):      # a country missing from Source 2
    if os.path.exists(f3.replace("test_source3_", "test_source2_")): continue
    g = pl.read_parquet(f3, columns=["s1_row", "cand_id"]).group_by("s1_row").agg(pl.col("cand_id").unique(maintain_order=True))
    for r, c in zip(g["s1_row"].to_list(), g["cand_id"].to_list()):
        cline[r] = ",".join(c)
    del g; gc.collect()
with open(f"{WORK}/output/candidate_pairs.tsv", "w", encoding="utf-8") as f:
    f.write("source1_entity_id\tcandidate_entity_ids\n")
    for s_, c in zip(s1_ids, cline):
        f.write(s_ + "\t" + c + "\n")
log("candidate file written")


In [ ]:
# candidate_pairs check is skipped (none.tsv): it needs a lot of memory (same as v1.2)
validator = glob.glob(f'{ROOT}/**/validate_submission.py', recursive=True)[0]
test_dir = os.path.dirname(find_file(ROOT, 'test_source1')[0])
!python3 {validator} --matching {WORK}/output/matching_results.tsv --candidate none.tsv --test-dir {test_dir}

**Done.** Open this version's **Output** tab, go into `output/`, and download **`matching_results.tsv`**. Keep `candidate_pairs.tsv` for the final zip. You can delete the `parts_v7/` folder.